In [1]:
%pip install -U ipywidgets widgetsnbextension

Note: you may need to restart the kernel to use updated packages.


In [2]:
# If needed:  %pip install plotly -q

import numpy as np
import plotly.graph_objects as go

In [3]:
import plotly.io as pio
# Choose ONE:
pio.renderers.default = "notebook_connected"  # Classic Jupyter Notebook (most reliable)
# pio.renderers.default = "jupyterlab"        # JupyterLab
# pio.renderers.default = "vscode"            # VS Code


In [4]:
# ----- 1) Toy data: y = 5 + 2.5x + noise -----
rng = np.random.default_rng(0)
n = 120
x = np.linspace(0, 10, n)
y = 5 + 2.5 * x + rng.normal(0, 2.0, size=n)

In [5]:

# ----- 2) Gradient Descent with a tiny bit of math safety -----
def gd_linreg_history(x, y, steps=400, lr=None):
    """
    Returns (weights_history, losses, (mu, sd))
      - We standardize x for stable GD: z = (x - mu)/sd
      - Model in z-space: y_hat = b0 + b1 * z
      - weights_history: array of [b0, b1] for each step
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mu, sd = x.mean(), x.std() + 1e-12
    z = (x - mu) / sd

    # Design matrix with bias
    Xb = np.c_[np.ones((len(z), 1)), z]

    # Auto step size if not provided (safe upper bound ~ 1/L)
    if lr is None:
        smax = np.linalg.svd(Xb, compute_uv=False)[0]  # largest singular value
        L = (2.0 / len(y)) * (smax ** 2)
        lr = 0.8 / L

    w = np.zeros(2)  # [b0, b1]
    W = [w.copy()]
    losses = [np.mean((Xb @ w - y) ** 2)]

    for _ in range(steps):
        yhat = Xb @ w
        grad = (2/len(y)) * (Xb.T @ (yhat - y))
        w = w - lr * grad
        W.append(w.copy())
        losses.append(np.mean((Xb @ w - y) ** 2))

    return np.array(W), np.array(losses), (mu, sd), lr

In [6]:
W, losses, (mu, sd), used_lr = gd_linreg_history(x, y, steps=20, lr=0.1)
print(f"Learning rate used: {used_lr:.4g}")

# Convert the final weights back to ORIGINAL x-units for curiosity:
b0_std, b1_std = W[-1]
w0 = b0_std - b1_std * (mu / sd)  # intercept in x-units
w1 = b1_std / sd                  # slope in x-units
print(f"Final line (original units): y ≈ {w0:.3f} + {w1:.3f}·x")

Learning rate used: 0.1
Final line (original units): y ≈ 4.976 + 2.497·x


In [7]:
W

array([[ 0.        ,  0.        ],
       [ 3.53248586,  1.47042935],
       [ 6.35847455,  2.64677283],
       [ 8.6192655 ,  3.58784762],
       [10.42789826,  4.34070744],
       [11.87480447,  4.94299531],
       [13.03232944,  5.42482559],
       [13.95834941,  5.81028983],
       [14.69916539,  6.11866121],
       [15.29181818,  6.36535832],
       [15.7659404 ,  6.56271601],
       [16.14523818,  6.72060216],
       [16.44867641,  6.84691108],
       [16.69142699,  6.94795821],
       [16.88562745,  7.02879592],
       [17.04098782,  7.09346609],
       [17.16527612,  7.14520222],
       [17.26470676,  7.18659113],
       [17.34425127,  7.21970225],
       [17.40788688,  7.24619115],
       [17.45879536,  7.26738227]])

In [8]:
# ----- 3) Build Plotly animation: line moves towards the fit -----
xs = np.linspace(x.min(), x.max(), 200)
frames_idx = np.unique(np.linspace(0, len(W)-1, 60, dtype=int))

# Initial line
b0, b1 = W[frames_idx[0]]
ys0 = b0 + b1 * ((xs - mu) / sd)

fig = go.Figure()

# Data
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="data", opacity=0.7))

# Moving fit line (initial)
fig.add_trace(go.Scatter(x=xs, y=ys0, mode="lines", name="fit (moving)"))

# Frames: update ONLY trace 1 (the moving line)
frames = []
for i in frames_idx:
    b0, b1 = W[i]
    ys = b0 + b1 * ((xs - mu) / sd)
    frames.append(
        go.Frame(
            data=[go.Scatter(x=xs, y=ys)],  # one item because we update one trace
            traces=[1],                     # trace index 1 = the line
            name=str(i)
        )
    )
fig.frames = frames


# Controls
fig.update_layout(
    title=f"Gradient Descent fitting a line (play me) — lr={used_lr:.3g}",
    xaxis_title="x",
    yaxis_title="y",
    updatemenus=[{
        "type": "buttons",
        "x": 0.02, "y": 1.12, "xanchor": "left",
        "buttons": [
            {"label": "▶ Play", "method": "animate",
             "args": [None, {"frame": {"duration": 400, "redraw": True},
                             "fromcurrent": True, "transition": {"duration": 200}}]},
            {"label": "⏸ Pause", "method": "animate",
             "args": [[None], {"frame": {"duration": 0, "redraw": False},
                               "mode": "immediate", "transition": {"duration": 0}}]}
        ]
    }],
    sliders=[{
        "x": 0.25, "y": 1.08, "xanchor": "left", "len": 0.6,
        "steps": [{"args": [[str(i)], {"frame": {"duration": 0, "redraw": True},
                                       "mode": "immediate"}],
                   "label": f"{i}", "method": "animate"} for i in frames_idx]
    }]
)
fig.show()

# ----- 4) Separate Plotly chart: loss curve (how error falls) -----
fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(x=np.arange(len(losses)), y=losses, mode="lines+markers", name="loss"))
fig_loss.update_layout(title="Mean Squared Error vs Iteration",
                       xaxis_title="iteration", yaxis_title="MSE")
fig_loss.show()

In [9]:
# If needed: %pip install scikit-learn plotly -q
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ---- Reuse from your GD cell ----
# x, y already defined
# W, losses, (mu, sd) already computed
xs = np.linspace(x.min(), x.max(), 200)

# GD final line in ORIGINAL units
b0_std, b1_std = W[-1]
w0_gd = b0_std - b1_std * (mu / sd)
w1_gd = b1_std / sd
y_gd_line = w0_gd + w1_gd * xs

print(f"GD final line: y ≈ {w0_gd:.3f} + {w1_gd:.3f}·x")

# ---- 2a) Fit scikit-learn Linear Regression (closed-form OLS) ----
lr = LinearRegression()
lr.fit(x.reshape(-1, 1), y)
w0_ols = lr.intercept_
w1_ols = lr.coef_[0]
y_ols_line = w0_ols + w1_ols * xs
mse_ols = mean_squared_error(y, lr.predict(x.reshape(-1, 1)))

print(f"OLS (sklearn) line: y ≈ {w0_ols:.3f} + {w1_ols:.3f}·x")
print(f"OLS in-sample MSE: {mse_ols:.3f}")

# ---- 2b) Plotly figure: data + GD final line + OLS line ----
fig_cmp = go.Figure()
fig_cmp.add_trace(go.Scatter(x=x, y=y, mode="markers", name="data", opacity=0.7))
fig_cmp.add_trace(go.Scatter(x=xs, y=y_gd_line, mode="lines", name="GD final"))
fig_cmp.add_trace(go.Scatter(x=xs, y=y_ols_line, mode="lines", name="OLS (sklearn)",
                             line=dict(dash="dash")))
fig_cmp.update_layout(
    title="Linear Regression: Gradient Descent (final) vs OLS (sklearn)",
    xaxis_title="x", yaxis_title="y"
)
fig_cmp.show()

# ---- 2c) Plotly loss chart: GD curve + OLS MSE reference ----
iters = np.arange(len(losses))
fig_loss_cmp = go.Figure()
fig_loss_cmp.add_trace(go.Scatter(x=iters, y=losses, mode="lines+markers", name="GD loss"))
# Draw OLS MSE as a horizontal reference line
fig_loss_cmp.add_trace(go.Scatter(x=[iters.min(), iters.max()],
                                  y=[mse_ols, mse_ols],
                                  mode="lines", name=f"OLS MSE = {mse_ols:.3f}",
                                  line=dict(dash="dash")))
fig_loss_cmp.update_layout(title="Loss vs Iteration (GD) with OLS optimum",
                           xaxis_title="iteration", yaxis_title="MSE")
fig_loss_cmp.show()


GD final line: y ≈ 4.976 + 2.497·x
OLS (sklearn) line: y ≈ 5.034 + 2.526·x
OLS in-sample MSE: 3.642


In [10]:
import numpy as np

# pretend we already have x, y from your notebook
Xb = np.c_[np.ones_like(x), x]   # [1, x]
w  = np.array([0.0, 0.0])        # start at C=0, M=0
yhat = Xb @ w
e = yhat - y

grad = (2/len(y)) * (Xb.T @ e)
dC, dM = grad
print("dC, dM =", dC, dM)

lr = 0.1
w_new = w - lr * grad
print("w ->", w, "updates to ->", w_new)


dC, dM = -35.32485861807107 -219.4271487241315
w -> [0. 0.] updates to -> [ 3.53248586 21.94271487]


In [11]:
e

array([ -5.25146044,  -4.94587431,  -6.70101337,  -5.84005234,
        -4.76899739,  -6.77361028,  -8.86850429,  -8.36475016,
        -5.2732018 ,  -4.35991336,  -5.85429141,  -7.39357633,
        -2.87094685,  -7.29350911,  -5.44935458,  -6.68672579,
        -7.27282657,  -7.93882826,  -9.60477368, -11.07662338,
        -8.94461135, -12.14469165,  -8.29145939, -10.53495291,
       -11.84895717, -10.44012544,  -8.97518638,  -8.82881816,
        -9.96690129, -11.53282722,  -9.28328464, -11.09425389,
       -11.40423906, -13.01446428, -12.57217539, -13.06368659,
       -11.25536799, -12.51388198, -14.55114422, -16.1801396 ,
       -10.88523028, -16.64129293, -16.51528026, -15.59623625,
       -14.77260874, -13.82593588, -17.57990691, -18.79446621,
       -18.68730335, -17.92432518, -16.2189625 , -13.29764845,
       -15.91546148, -17.44740365, -13.76781489, -17.34486597,
       -17.62443327, -18.36687536, -14.81663802, -16.07155284,
       -16.73217152, -15.47552223, -21.50394584, -17.24

In [12]:
Xb.T

array([[ 1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1.        ,  1.        ,
         1.        ,  1.        ,  1.        ,  1. 

In [13]:
Xb.T @ e

array([ -2119.49151708, -13165.62892345])

In [14]:
# %pip install plotly ipywidgets --quiet

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import HBox, VBox, Dropdown, FloatText, IntSlider, Checkbox, Button, Layout, HTML

# ---------- Toy data ----------
rng = np.random.default_rng(0)
n = 120
x = np.linspace(0, 10, n)
y = 5 + 2.5 * x + rng.normal(0, 2.0, size=n)

# ---------- Helpers ----------
def add_bias(v):
    return np.c_[np.ones((len(v), 1)), v]

def auto_lr(Xb, n):
    # Safe step size: lr < 1/L ; L = (2/n) * s_max^2
    smax = np.linalg.svd(Xb, compute_uv=False)[0]
    L = (2.0 / n) * (smax**2)
    return 0.8 / L

def gd_history(x, y, steps, lr, standardize=True, init="zeros"):
    """
    Trains y ≈ C + M*x using GD in *either* z-space (standardize=True) or x-space.
    Returns dict with histories in original x-units:
      w0_hist, w1_hist, loss_hist, grads_hist[(dC,dM) in original units], (mu,sd), used_lr
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    n = len(y)

    if standardize:
        mu, sd = x.mean(), x.std() + 1e-12
        z = (x - mu) / sd
        Xb = add_bias(z)
    else:
        mu, sd = 0.0, 1.0
        Xb = add_bias(x)

    # init in the space we optimize in (b0,b1)
    if init == "zeros":
        b = np.zeros(2)
    else:
        b = np.array([0.0, 0.0]) + rng.normal(0, 0.1, size=2)

    if lr is None:
        lr = auto_lr(Xb, n)

    w0_hist, w1_hist, loss_hist = [], [], []
    grads_hist = []  # (dC, dM) in original x-units

    for _ in range(max(1, steps)):
        # Save current params converted to original x units
        # If trained in z: y = b0 + b1*((x-mu)/sd) = (b0 - b1*mu/sd) + (b1/sd) * x
        if standardize:
            C = b[0] - b[1] * (mu / sd)
            M = b[1] / sd
            # Residuals computed in training space
            yhat = Xb @ b
        else:
            C, M = b[0], b[1]
            yhat = Xb @ b

        w0_hist.append(C)
        w1_hist.append(M)
        loss_hist.append(np.mean((yhat - y)**2))

        # Gradient in training space (b0,b1)
        e = yhat - y
        grad_b = (2.0/n) * (Xb.T @ e)  # [d/db0, d/db1]

        # Convert gradient to original x-units for display: dL/dC = dL/db0 ; dL/dM = (dL/db1)*(db1/dM)
        # From M = b1/sd  => db1/dM = sd ; and C = b0 - b1*mu/sd so dC/db0 = 1, dC/db1 = -mu/sd (not needed for display)
        dC = grad_b[0]                             # same as d/db0
        dM = grad_b[1] * sd if standardize else grad_b[1]
        grads_hist.append((float(dC), float(dM)))

        # Step
        b = b - lr * grad_b

    return {
        "w0_hist": np.array(w0_hist),
        "w1_hist": np.array(w1_hist),
        "loss_hist": np.array(loss_hist),
        "grads_hist": np.array(grads_hist),
        "mu_sd": (mu, sd),
        "used_lr": lr
    }

def closed_form_line(x, y):
    Xb = add_bias(x)
    w = np.linalg.pinv(Xb.T @ Xb) @ (Xb.T @ y)
    return w[0], w[1]  # C, M

# ---------- Build FigureWidget ----------
xs = np.linspace(x.min(), x.max(), 200)
C_ols, M_ols = closed_form_line(x, y)
y_ols = C_ols + M_ols * xs

fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
                    subplot_titles=("GD fit (line updates)", "Loss vs iteration"))

fig = go.FigureWidget(fig)
# Data scatter (left)
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="data", opacity=0.7), row=1, col=1)
# OLS line (left, dashed)
fig.add_trace(go.Scatter(x=xs, y=y_ols, mode="lines", name="OLS (closed-form)",
                         line=dict(dash="dash")), row=1, col=1)
# GD moving line (left) – placeholder
line_gd = go.Scatter(x=xs, y=xs*0, mode="lines", name="GD line")
fig.add_trace(line_gd, row=1, col=1)
# Loss curve (right) – placeholder
loss_curve = go.Scatter(x=[0], y=[0], mode="lines+markers", name="loss")
fig.add_trace(loss_curve, row=1, col=2)

fig.update_xaxes(title="x", row=1, col=1)
fig.update_yaxes(title="y", row=1, col=1)
fig.update_xaxes(title="iteration", row=1, col=2)
fig.update_yaxes(title="MSE", row=1, col=2)
fig.update_layout(height=520, width=1000, title="Linear Regression via Gradient Descent (interactive)")

# ---------- Controls ----------
lr_dropdown = Dropdown(
    options=[("Auto (safe)", "auto"), ("0.01", 0.01), ("0.05", 0.05), ("0.1", 0.1), ("0.2", 0.2)],
    value="auto", description="Learning rate:", layout=Layout(width="270px")
)
steps_slider = IntSlider(description="Max steps:", min=5, max=500, step=5, value=100,
                         continuous_update=False, readout=True, layout=Layout(width="400px"))
iter_slider = IntSlider(description="Iteration:", min=1, max=100, step=1, value=1,
                        continuous_update=True, readout=True, layout=Layout(width="400px"))
std_check = Checkbox(value=True, description="Standardize x (z-score)")
init_dropdown = Dropdown(options=[("Zeros", "zeros"), ("Random small", "random")],
                         value="zeros", description="Initial W:")
status = HTML("<b>Status:</b> —")

def recompute_and_refresh(*_):
    # Resolve lr
    lr_val = None if lr_dropdown.value == "auto" else float(lr_dropdown.value)
    steps = int(steps_slider.value)

    hist = gd_history(x, y, steps=steps, lr=lr_val,
                      standardize=std_check.value, init=init_dropdown.value)

    # Save on figure object for access in other callbacks
    fig._hist = hist

    # Reset iteration slider range and value
    iter_slider.max = steps
    iter_slider.value = 1  # start at first update

    # Update once for iteration=1
    update_iteration()

def update_iteration(*_):
    if not hasattr(fig, "_hist"):
        return
    t = int(iter_slider.value) - 1  # 0-based index
    h = fig._hist
    C = h["w0_hist"][t]
    M = h["w1_hist"][t]
    losses = h["loss_hist"]
    dC, dM = h["grads_hist"][t]
    lr_used = h["used_lr"]

    # Update GD line
    y_line = C + M * xs
    with fig.batch_update():
        fig.data[2].x = xs
        fig.data[2].y = y_line
        fig.data[2].name = f"GD line (iter {t+1})"

        # Update loss curve up to t
        fig.data[3].x = np.arange(t+1)
        fig.data[3].y = losses[:t+1]

        fig.layout.title = (f"GD (lr={lr_used:.4g}, standardize={std_check.value}, init={init_dropdown.value}) — "
                            f"iter {t+1}/{iter_slider.max}")

    status.value = (f"<b>Status:</b> iter={t+1}, "
                    f"C={C:.3f}, M={M:.3f}, "
                    f"gradC={dC:.3f}, gradM={dM:.3f}, "
                    f"loss={losses[t]:.3f}")

# Wire events
for w in (lr_dropdown, steps_slider, std_check, init_dropdown):
    w.observe(recompute_and_refresh, names="value")
iter_slider.observe(update_iteration, names="value")

# First render
recompute_and_refresh()

VBox([
    fig,
    HBox([lr_dropdown, std_check, init_dropdown]),
    HBox([steps_slider, iter_slider]),
    status
])


    'data': [{'mode': 'markers',
              'name': 'data',
              'op…

In [15]:
# Install (safe to re-run)
# %pip install -q plotly ipywidgets

import plotly, plotly.io as pio
print("plotly:", plotly.__version__)
print("available renderers:", list(pio.renderers.keys())[:6], "...")

# Pick ONE renderer that matches your setup:
# Jupyter Notebook (most reliable):
pio.renderers.default = "notebook_connected"
# If you're offline: pio.renderers.default = "notebook"
# VS Code:          pio.renderers.default = "vscode"
# JupyterLab 3+:    pio.renderers.default = "jupyterlab"

# Widgets check
import ipywidgets as widgets
from IPython.display import display, HTML
print("ipywidgets:", widgets.__version__)

# Classic Notebook only (optional but helps if widgets still don't show):
# !jupyter nbextension enable --py widgetsnbextension --sys-prefix
# !jupyter nbextension list

# Google Colab only (uncomment both lines if using Colab):
# %pip install -q ipywidgets
# from google.colab import output; output.enable_custom_widget_manager()


plotly: 6.3.0
available renderers: ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode', 'notebook', 'notebook_connected'] ...
ipywidgets: 8.1.7


In [16]:
from IPython.display import display

ui = VBox([
    fig,
    HBox([lr_dropdown, std_check, init_dropdown]),
    HBox([steps_slider, iter_slider]),
    status
])
display(ui)  # <-- force render


    'data': [{'mode': 'markers',
              'name': 'data',
              'op…

In [21]:
import plotly.io as pio
# Choose ONE:
pio.renderers.default = "notebook_connected"  # Classic Jupyter Notebook (most reliable)
# pio.renderers.default = "jupyterlab"        # JupyterLab
# pio.renderers.default = "vscode"            # VS Code


In [22]:
import ipywidgets as widgets
widgets.IntSlider()  # should render a slider, not a text repr


IntSlider(value=0)

In [23]:
from IPython.display import display
ui = VBox([fig,
           HBox([lr_dropdown, std_check, init_dropdown]),
           HBox([steps_slider, iter_slider]),
           status])
display(ui)


    'data': [{'mode': 'markers',
              'name': 'data',
              'op…